# CYMEK CS-TRANSFER-001 A2 — hardened one-click T4 run

Run the single cell below in the current T4 session. Do **not** restart the runtime. Existing Drive state is preserved.

This launcher keeps the scientific executable frozen at `f8582808b6e2be0753cb2689d9d6d4aeb4d57aeb`, uses `MyDrive/CYMEK/CS_TRANSFER_001_A2`, repairs the contradictory tiny serialization qualification fixture, avoids the irrelevant 250M model-construction test, and resumes valid A2 checkpoints.


In [ ]:
from google.colab import drive
import pathlib
if not pathlib.Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)

import hashlib, subprocess, sys, time, urllib.request

OPERATOR_COMMIT = '4b2ca58f63c0ea4b0e36dc38f51b4133864808fc'
FILES = {
    'cs_transfer_001_colab_operator_v4.py': 'a1cb5f9f1f642684f85be8d70585ac382920a85a',
    'cs_transfer_001_colab_operator_v5.py': 'a1a6cafcb2f00bd39a121bdfb931875a1ee6c02b',
}
BASE = 'https://raw.githubusercontent.com/dhurv0045com-spec/An-Ra-the-new-AGI/' + OPERATOR_COMMIT + '/tools/'

def fetch_verified(name, expected_blob):
    url = BASE + name
    payload = None
    last = None
    for attempt in range(1, 4):
        try:
            req = urllib.request.Request(url, headers={'Cache-Control': 'no-cache'})
            with urllib.request.urlopen(req, timeout=60) as response:
                payload = response.read()
            break
        except Exception as exc:
            last = exc
            print(f'{name} download attempt {attempt}/3 failed: {exc}')
            time.sleep(2 * attempt)
    if payload is None:
        raise RuntimeError(f'Could not download {name}: {last}')
    got = hashlib.sha1(f'blob {len(payload)}\0'.encode() + payload).hexdigest()
    if got != expected_blob:
        raise RuntimeError(f'{name} identity mismatch: {got} != {expected_blob}')
    dest = pathlib.Path('/content') / name
    dest.write_bytes(payload)
    print('VERIFIED:', name, got)
    return dest

for name, blob in FILES.items():
    fetch_verified(name, blob)

print('OPERATOR: v5 hardened qualification repair')
print('SCIENTIFIC EXECUTABLE remains frozen at f8582808b6e2be0753cb2689d9d6d4aeb4d57aeb')
subprocess.run([sys.executable, '-u', '/content/cs_transfer_001_colab_operator_v5.py'], check=True)
